In [12]:

from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd

from deeprootgen.data_model import RootSimulationModel
from deeprootgen.model import RootSystemSimulation
from deeprootgen.statistics.root_grid_aggregation import (
    aggregate_root_grid,
)

# --------------------------------------------------
# Configuration
# --------------------------------------------------

INFILE = (
    Path("..", "..", "data", "grapevine")
    / "fichtl_2024_voxel_grid.csv"
)

OUT_DIR = Path("outputs/grapevine_calibration")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Start with one rootstock-timepoint group.
GROUPS_TO_FIT = [("101-14", "T1")]

# Use this instead to fit all 12 groups.
# GROUPS_TO_FIT = None

VOXEL_SIZE_CM = 10.0

# Increase for a more robust stochastic calibration.
N_REPLICATES = 2

# Eight vines per group: six training, two testing.
N_TEST_VINES = 2

BASE_SEED = 42
SHOW_ROOT_PLOT = False

# Initial small one-shot parameter grid.
PARAMETER_GRID = {
    "max_primary_length": [25.0, 40.0, 55.0],
    "max_sec_root_num": [3, 4],
}

# Observed root length by branching order.
ORDER_COLS = {
    1: "primary_length_cm",
    2: "secondary_length_cm",
    3: "tertiary_length_cm",
    4: "quaternary_length_cm",
    5: "quinary_length_cm",
}

# --------------------------------------------------
# Baseline model parameters
# --------------------------------------------------

BASE_PARAMETERS = dict(
    random_seed=BASE_SEED,
    max_order=3,
    root_ratio=0.5,
    fine_root_threshold=0.15,
    outer_root_num=4,
    inner_root_num=2,
    min_primary_length=15.0,
    max_primary_length=40.0,
    base_diameter=0.5,
    diameter_reduction=0.2,
    apex_diameter=0.05,
    min_sec_root_num=1,
    max_sec_root_num=3,
    growth_sec_root=0.1,
    min_sec_root_length=3.0,
    max_sec_root_length=15.0,
    segments_per_root=6,
    length_reduction=0.7,
    root_vary=10.0,
    interbranch_distance=2.0,
    mechanical_constraints=0.0,
    root_tissue_density=0.15,
    gravitropism=0.5,
    t=1,
    enable_soil=False,
    soil_layer_height=10,
    soil_n_layers=10,
    soil_layer_width=10,
    soil_n_cols=10,
)

# --------------------------------------------------
# Load and validate observations
# --------------------------------------------------

observed_df = pd.read_csv(INFILE)

required = [
    "timepoint",
    "vine_id",
    "rootstock",
    "x_min_cm",
    "y_min_cm",
    "depth_min_cm",
    "root_length_cm",
    "rld_cm_per_cm3",
    *ORDER_COLS.values(),
    "other_length_cm",
]

missing = set(required) - set(observed_df.columns)

if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

if observed_df[required].isna().any().any():
    raise ValueError("Missing values in required columns.")

# Check that RLD and root length are consistent.
np.testing.assert_allclose(
    observed_df["root_length_cm"],
    observed_df["rld_cm_per_cm3"] * VOXEL_SIZE_CM**3,
    rtol=1e-5,
    atol=1e-5,
)

# Check that branching-order lengths sum to total length.
np.testing.assert_allclose(
    observed_df["root_length_cm"],
    observed_df[
        [*ORDER_COLS.values(), "other_length_cm"]
    ].sum(axis=1),
    rtol=1e-5,
    atol=1e-5,
)

print("Dataset validation passed.")

print(f"Observations: {len(observed_df):,}")
print(f"Vines: {observed_df['vine_id'].nunique()}")
print(f"Rootstocks: {observed_df['rootstock'].nunique()}")
print(f"Timepoints: {observed_df['timepoint'].nunique()}")

display(
    observed_df.groupby(
        ["rootstock", "timepoint"]
    ).agg(
        vines=("vine_id", "nunique"),
        voxels=("vine_id", "count"),
        occupied_voxels=(
            "rld_cm_per_cm3",
            lambda x: (x > 0).sum(),
        ),
        total_root_length_cm=("root_length_cm", "sum"),
    )
)


# Summarise each vine before aggregating by rootstock and timepoint.
vine_summary = (
    observed_df.groupby(
        ["rootstock", "timepoint", "vine_id"]
    )
    .agg(
        voxels=("rld_cm_per_cm3", "size"),
        occupied_voxels=(
            "rld_cm_per_cm3",
            lambda x: (x > 0).sum(),
        ),
        total_root_length_cm=("root_length_cm", "sum"),
        mean_rld=("rld_cm_per_cm3", "mean"),
    )
    .reset_index()
)

vine_summary["occupancy_fraction"] = (
    vine_summary["occupied_voxels"]
    / vine_summary["voxels"]
)

group_summary = (
    vine_summary.groupby(["rootstock", "timepoint"])
    .agg(
        vines=("vine_id", "nunique"),
        mean_voxels=("voxels", "mean"),
        mean_occupancy=("occupancy_fraction", "mean"),
        mean_root_length_cm=("total_root_length_cm", "mean"),
        mean_rld=("mean_rld", "mean"),
    )
    .reset_index()
)

display(group_summary.round(4))


Dataset validation passed.
Observations: 63,653
Vines: 96
Rootstocks: 3
Timepoints: 4


vines  voxels  occupied_voxels  total_root_length_cm
rootstock timepoint                                                      
101-14    T1             8    1077              336           7428.216033
          T2             8    4459              827          14640.905108
          T3             8    3810              693          12922.947345
          T4             8   10415             2007          33724.179317
R110      T1             8     748              280           7575.035070
          T2             8    2486              722          14492.663307
          T3             8    3965              913          19194.986435
          T4             8    7715             2010          45104.563236
SO4       T1             8    1031              291           6086.618209
          T2             8    5502              851          14230.780171
          T3             8    5937              908          13563.896085
          T4             8   16508             2545          38514.083389

,rootstock,timepoint,vines,mean_voxels,mean_occupancy,mean_root_length_cm,mean_rld
0,101-14,T1,8,134.625,0.3433,928.5270,0.0081
1,101-14,T2,8,557.375,0.1948,1830.1131,0.0036
2,101-14,T3,8,476.250,0.2596,1615.3684,0.0066
3,101-14,T4,8,1301.875,0.1922,4215.5224,0.0032
4,R110,T1,8,93.500,0.3885,946.8794,0.0107
5,R110,T2,8,310.750,0.3206,1811.5829,0.0065
6,R110,T3,8,495.625,0.2612,2399.3733,0.0057
7,R110,T4,8,964.375,0.2852,5638.0704,0.0066
8,SO4,T1,8,128.875,0.2869,760.8273,0.0060
9,SO4,T2,8,687.750,0.1789,1778.8475,0.0032
